In [ ]:
import sys
from pathlib import Path

# Make dissector importable without a formal install
sys.path.insert(0, str(Path('..') / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from dissector.evaluation import radarplot_single_dataset

DATA_DIR    = Path('final_results_copy')
SUMMARY_DIR = Path('summary_results')


In [ ]:
!pwd

In [ ]:
# Load per-muscle MyoSegmenTUM results from summary_results/
#
# Each CSV has one row per muscle (L/R gracilis, L/R sartorius) averaged over
# all subjects. Water and fat-fraction results are averaged together so each
# algorithm ends up with four rows -- one per muscle.

MUSCLE_COLORS = {
    'L_gracilis':  '#1f77b4',
    'R_gracilis':  '#4a90d9',
    'L_sartorius': '#ff7f0e',
    'R_sartorius': '#f5a623',
}

RADAR_COLS   = ['dice', 'jaccard', 'boundary_iou_3d', 'recall']
RADAR_LABELS = ['Dice', 'Jaccard\n(IoU)', 'Boundary\nIoU', 'Recall\n(1\u2212FN)']


def build_myo_radar_df(water_csv: Path, fatfrac_csv: Path) -> pd.DataFrame:
    """Return a DataFrame indexed by muscle with normalised radar metrics."""
    dfs = []
    for p in (water_csv, fatfrac_csv):
        if p.exists():
            df = pd.read_csv(p, index_col=0)
            dfs.append(df[df.index != 'Overall_Mean'])
        else:
            print(f'Warning: {p} not found, skipping.')
    if not dfs:
        raise FileNotFoundError('No summary CSVs found.')
    combined = pd.concat(dfs)
    grouped  = combined.groupby(combined.index)[['dice', 'jaccard', 'boundary_iou_3d', 'false_negative']].mean()
    grouped['recall'] = 1.0 - grouped['false_negative']
    return grouped[RADAR_COLS]


df_wb    = build_myo_radar_df(
    SUMMARY_DIR / 'musclemap_wb_water_avg_metrics.csv',
    SUMMARY_DIR / 'musclemap_wb_fat_fraction_avg_metrics.csv',
)
df_thigh = build_myo_radar_df(
    SUMMARY_DIR / 'musclemap_thigh_water_avg_metrics.csv',
    SUMMARY_DIR / 'musclemap_thigh_fat_fraction_avg_metrics.csv',
)

print('MuscleMap WB:\n',    df_wb.round(3))
print('\nMuscleMap Thigh:\n', df_thigh.round(3))


In [ ]:
# Quick preview using radarplot_single_dataset from dissector.evaluation
# (Each row becomes one spider line; all plotted in blue.)

radarplot_single_dataset(
    metrics=RADAR_COLS,
    dataset=df_wb.reset_index(),
    title='MuscleMap WB -- per-muscle profiles (MyoSegmenTUM)',
    labels=RADAR_LABELS,
)

radarplot_single_dataset(
    metrics=RADAR_COLS,
    dataset=df_thigh.reset_index(),
    title='MuscleMap Thigh -- per-muscle profiles (MyoSegmenTUM)',
    labels=RADAR_LABELS,
)


In [ ]:
# Side-by-side coloured radar -- MuscleMap WB (left) vs MuscleMap Thigh (right)
# Each muscle gets its own colour; both panels share a legend.
# Black dashed line = mean across muscles.

PLOT_COLS   = ['dice', 'jaccard', 'boundary_iou_3d', 'recall']
PLOT_LABELS = ['Dice', 'Jaccard\n(IoU)', 'Boundary\nIoU', 'Recall\n(1\u2212FN)']

num_metrics = len(PLOT_COLS)
angles = np.linspace(0, 2 * np.pi, num_metrics, endpoint=False)
angles = np.append(angles, angles[0])

fig, axes = plt.subplots(1, 2, figsize=(13, 6), subplot_kw={'polar': True})
fig.subplots_adjust(wspace=0.45)

panels = [
    (axes[0], df_wb,    'MuscleMap WB (v2.0)'),
    (axes[1], df_thigh, 'MuscleMap Thigh (v2.0)'),
]

for ax, df, title in panels:
    for muscle_name, row in df.iterrows():
        vals  = row[PLOT_COLS].to_numpy(dtype=float)
        vals  = np.append(vals, vals[0])
        color = MUSCLE_COLORS.get(muscle_name, '#888888')
        ax.plot(angles, vals, 'o-', linewidth=1.8, color=color, alpha=0.85, markersize=4)
        ax.fill(angles, vals, alpha=0.07, color=color)

    mean_vals = df[PLOT_COLS].mean().to_numpy(dtype=float)
    mean_vals = np.append(mean_vals, mean_vals[0])
    ax.plot(angles, mean_vals, linewidth=2.5, color='black', linestyle='--',
            alpha=0.6, zorder=5)
    ax.fill(angles, mean_vals, alpha=0.12, color='black')

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(PLOT_LABELS, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.50, 0.75, 1.0])
    ax.set_yticklabels(['0.25', '0.50', '0.75', '1.0'], fontsize=7, color='grey')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=18)
    ax.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.5)

handles = [
    mpatches.Patch(color=MUSCLE_COLORS[m], label=m.replace('_', ' '))
    for m in MUSCLE_COLORS
    if m in df_wb.index or m in df_thigh.index
]
handles.append(plt.Line2D([0], [0], color='black', linestyle='--',
                           linewidth=2.5, alpha=0.6, label='Mean'))

fig.legend(handles=handles, title='Muscle', loc='lower center',
           ncol=len(handles), fontsize=9, framealpha=0.9,
           bbox_to_anchor=(0.5, -0.04))

fig.suptitle(
    'MuscleMap WB vs MuscleMap Thigh -- Per-muscle Metric Profiles (MyoSegmenTUM)',
    fontsize=12, fontweight='bold', y=1.02
)

plt.savefig('radar_musclemap_wb_vs_thigh.pdf', bbox_inches='tight')
plt.savefig('radar_musclemap_wb_vs_thigh.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved radar_musclemap_wb_vs_thigh.pdf / .png')
